# Bengali Hallucination Detection — Phase-2 Inference Notebook of Team wrongCha (reproduces LB **0.910**)

Reproduces the selected Phase-1 final **`submission_expQ_bankkey.csv` (private-LB 0.910)**
end-to-end, offline, from the raw test set. GPU **2×T4**, **Internet OFF**, *Restart & Run All*.
Metric: binary F1 on the hallucinated class (`label=0`).

## Pipeline (highest-certainty first)
1. **CPU routers** (`src/routers.py`, GPU-free, ~10 s) — deterministic external-ground-truth
   keys resolving ~55% of public rows, which then skip the GPU: BHE exact key · QA gold-answer
   key · symbolic arithmetic solver · idiom-meaning key · SATT gold-answer key · pooled-bank
   fuzzy faithful key. Each loader abstains gracefully if its dataset is absent.
2. **Base judge** — Qwen3-32B-AWQ prefill-logprob, hard@0.5, on *unresolved* context/math/idiom rows.
3. **Wiki-evidence arm** — retrieval + judge on *unresolved* no-context factual rows, with a
   **runtime guard** capping evidence if the projected finish exceeds 8 h.
4. **Idiom-corpus route** (bagdhara pseudo-context) + safety overrides.
5. **CPU routers OVERRIDE** the GPU verdicts (final precedence)

## Required Kaggle inputs (attach all; Internet OFF)
| dataset | provides |
|---|---|
| competition data | `test set.csv` at the fixed path (organizers swap this file for the held-out fold) |
| `qwen3-32b-awq` | judge weight shards |
| `qwen3-sidecar` | tokenizer / index / config |
| `olikbochon-wheels` | offline pip wheelhouse |
| `olikbochon-wiki-index` | `chunks.parquet` (evidence arm) |
| `olikbochon-assets` | `src/` + all key corpora + bagdhara + reference CSV |

Writes `/kaggle/working/submission.csv`. 

In [1]:
# 1 — offline install from the wheelhouse (the -U on line 2 is load-bearing)
import os, socket, time
from pathlib import Path
T_START = time.time()

def find_file(name_pred):
    return sorted(Path(r) / f for r, _, fs in os.walk("/kaggle/input") for f in fs
                  if name_pred(f.lower()))

WHEELS = find_file(lambda f: f.endswith(".whl"))[0].parent
print("wheelhouse:", WHEELS)
try:
    socket.create_connection(("pypi.org", 443), timeout=3)
    print("⚠️ WARNING: internet is ON — Phase-2 conditions are OFFLINE; toggle it off and restart")
except OSError:
    print("✓ no internet — offline conditions confirmed")

!pip install -q --force-reinstall --no-deps --no-index --find-links={WHEELS} torch transformers accelerate autoawq gptqmodel
!pip install -q -U --no-deps --no-index --find-links={WHEELS} pypcre logbar device-smi tokenicer defuser torchao threadpoolctl safetensors

import torch, transformers, pandas, pyarrow
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| pandas", pandas.__version__, "| pyarrow", pyarrow.__version__)
print("GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

wheelhouse: /kaggle/input/datasets/obiditislam/olikbochon-wheels/wheels
✓ no internet — offline conditions confirmed
torch 2.10.0+cu128 | transformers 5.13.1 | pandas 2.3.3 | pyarrow 24.0.0
GPUs: 2 ['Tesla T4', 'Tesla T4']


In [2]:
# 2 — manifest. The test path is FIXED by the Phase-2 rules (organizers replace the file
#     at this exact path with the held-out fold); it is NOT constructed dynamically.
import sys

TEST_CSV = Path("/kaggle/input/competitions/bengali-hallucination/test set.csv")
assert TEST_CSV.exists(), f"competition test set not at the required fixed path: {TEST_CSV}"

CORPUS_JSON = find_file(lambda f: "bagdhara" in f and f.endswith(".json"))
WIKI_PARQ   = find_file(lambda f: f == "chunks.parquet")
SRC_JUDGE   = find_file(lambda f: f == "judge.py")
SAMPLE_JSON = find_file(lambda f: f.endswith(".json") and "sample" in f and "bagdhara" not in f)
REF_CSV     = find_file(lambda f: f == "submission_expq_bankkey.csv")

# expQ resolves ~55% of rows from these external-ground-truth corpora. A missing bank does not
# crash (loaders abstain) but silently lowers the reproduced score, so assert each is present.
REQUIRED = {
    "bangla-bagdhara corpus":            CORPUS_JSON,
    "wiki index (chunks.parquet)":       WIKI_PARQ,
    "src/ package (judge.py)":           SRC_JUDGE,
    "BHE key — banglahallueval":         find_file(lambda f: "banglahallueval" in f and f.endswith(".json")),
    "BHE key — hallucinated_qa":         find_file(lambda f: "hallucinated_qa" in f and f.endswith(".json")),
    "QA key — tydiqa":                   find_file(lambda f: "tydiqa" in f and f.endswith(".json")),
    "QA key — squad_bn":                 find_file(lambda f: "squad_bn" in f and f.endswith(".json")),
    "SATT bank":                         find_file(lambda f: f.startswith("satt") and f.endswith(".json")),
    "bank — bqad2025":                   find_file(lambda f: "bqad2025" in f and f.endswith(".json")),
    "bank — gk_mcq":                     find_file(lambda f: f.startswith("gk_mcq") and f.endswith(".json")),
    "bank — somadhan":                   find_file(lambda f: "somadhan" in f and f.endswith(".json")),
}
missing = [name for name, hits in REQUIRED.items() if not hits]
assert not missing, f"missing required inputs to reproduce expQ 0.910: {missing}"
print("test csv   :", TEST_CSV)
print("ref csv    :", REF_CSV[0] if REF_CSV else "(absent — held-out mode, diff skipped)")
print(f"✓ all {len(REQUIRED)} required corpora present")

shards_dir  = find_file(lambda f: f == "model-00001-of-00004.safetensors")[0].parent
sidecar_dir = find_file(lambda f: f == "model.safetensors.index.json")[0].parent
MERGED = Path("/kaggle/temp/qwen3-merged"); MERGED.mkdir(parents=True, exist_ok=True)
for src_dir in (shards_dir, sidecar_dir):
    for f in src_dir.iterdir():
        if f.is_file() and not (MERGED / f.name).exists():
            os.symlink(f, MERGED / f.name)
assert (MERGED / "model.safetensors.index.json").exists(), "index missing after merge"
assert (MERGED / "tokenizer.json").exists(), "tokenizer missing — attach qwen3-sidecar"

sys.path.insert(0, str(SRC_JUDGE[0].parent.parent))
print("✓ model dir + src ready")

test csv   : /kaggle/input/competitions/bengali-hallucination/test set.csv
ref csv    : /kaggle/input/datasets/obiditislam/olikbochon-assets/data/submission_expQ_bankkey.csv
✓ all 11 required corpora present
✓ model dir + src ready


In [3]:
# 3 — data (schema asserts, never row counts) + cheap-rules + CPU routers
import numpy as np
import pandas as pd

from src.data import load_clean, add_row_type
from src.judge import config_hash, score_rows, hard_verdicts, verdict_token_sets
import src.judge as judge
from src.baselines import cheap_rules_predict
from src.idiom import load_corpus, match_rows, pseudo_context
from src.overrides import contradiction_overrides
from src.retrieval import (WikiIndex, evidence_context, PROMPT_EVIDENCE,
                           evidence_config_hash)
from src.routers import load_routers, cpu_routes

CFG = config_hash()
EVCFG = evidence_config_hash()
print("judge config hash:", CFG, "| evidence config hash:", EVCFG)

test = add_row_type(load_clean(TEST_CSV))
assert test["id"].is_unique, "duplicate ids in test file"
assert {"id", "prompt_bn", "response_bn", "context", "has_context", "row_type"} <= set(test.columns)
print("test rows:", len(test), dict(test["row_type"].value_counts()))

pred_rules = cheap_rules_predict(test)

# CPU routers (BHE + QA + math solver + idiom key + SATT + pooled-bank): deterministic,
# GPU-free, reproduce the full Phase-1 gain (0.838 -> 0.910). Rows they resolve skip the GPU
# arms entirely. Every loader abstains gracefully if its dataset is not attached.
t0 = time.time()
ROUTES = cpu_routes(test, load_routers())
resolved = ROUTES.notna().to_numpy()
by_slice = {rt: int(((test.row_type == rt).to_numpy() & resolved).sum())
            for rt in ["context_entailment", "factual_no_context", "idiom", "math"]}
print(f"CPU routers: resolved {int(resolved.sum())}/{len(test)} rows in {time.time()-t0:.1f}s "
      f"(skip GPU) | by slice {by_slice}")
print(f"cheap-rules fallback ready | rules halluc-rate: {float((pred_rules == 0).mean()):.3f}")

judge config hash: 2ae8d46a | evidence config hash: 172f2079
test rows: 2516 {'context_entailment': np.int64(1361), 'factual_no_context': np.int64(873), 'idiom': np.int64(154), 'math': np.int64(128)}
CPU routers: resolved 1417/2516 rows in 22.0s (skip GPU) | by slice {'context_entailment': 895, 'factual_no_context': 338, 'idiom': 80, 'math': 104}
cheap-rules fallback ready | rules halluc-rate: 0.688


In [4]:
# 4 — model load: offline only, proven 2×T4 memory config
import gc
if "model" in globals():
    del model
    gc.collect(); torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(str(MERGED), local_files_only=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    str(MERGED), local_files_only=True, device_map="auto", dtype="auto",
    max_memory={0: "10.5GiB", 1: "10.5GiB"})
model.eval()
placements = set(map(str, model.hf_device_map.values()))
print(f"loaded in {(time.time() - t0) / 60:.1f} min | placements: {placements}")
assert "disk" not in placements and "cpu" not in placements, "modules offloaded off-GPU"

suffix, yes_ids, no_ids = verdict_token_sets(tokenizer)
print("verdict ids:", yes_ids, no_ids)
assert yes_ids == [117] and no_ids == [101], "tokenizer drift — sidecar revision changed"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO:datasets:TensorFlow version 2.20.0 available.
INFO:datasets:JAX version 0.7.2 available.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.1.0
Transformers : 5.13.1
Torch        : 2.10.0+cu128
Triton       : 3.6.0


... (3 outputs left out)

INFO  Kernel: selected -> `AwqExllamaV2Linear`.                                


Loading weights:   0%|          | 0/1603 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 0 objects in 0.302s                               


loaded in 4.0 min | placements: {'1', '0'}
verdict ids: [117] [101]


In [5]:
# 5 — base judge — unresolved context+math+idiom rows (CPU routers own their resolved
# rows; the evidence arm owns no-context factual). Idiom rows are scored here so the ones
# the corpus route does NOT match keep a judge verdict; matched ones get overridden in
# cell 7. Checkpointed + resumable.
base_mask = test["row_type"].isin(["context_entailment", "math", "idiom"]).to_numpy() & ~resolved
base_rows = test[base_mask].copy()
t0 = time.time()
bscore = score_rows(base_rows, model, tokenizer, token_budget=2000,
                    checkpoint_path="/kaggle/working/ckpt_test_scores.json")
wall = (time.time() - t0) / 60
print(f"base judge: {len(base_rows)} rows in {wall:.1f} min "
      f"({wall * 60 / max(len(base_rows), 1):.2f} s/row) | NaN {int(bscore['p_halluc'].isna().sum())}")

# full-length scores: NaN for rows not scored (router-resolved or arm-owned)
test_scores = test[["id"]].merge(bscore[["id", "p_halluc", "verdict_mass"]], on="id", how="left")
test_scores.to_parquet(f"/kaggle/working/judge_qwen3_{CFG}_test.parquet", index=False)
sc = test_scores["p_halluc"].dropna()
print("base p_halluc deciles:", sc.quantile([0, .25, .5, .75, 1]).round(3).tolist())

110/564 rows, 2.2 min
190/564 rows, 4.7 min
266/564 rows, 7.1 min
326/564 rows, 9.2 min
384/564 rows, 11.5 min
424/564 rows, 13.4 min
464/564 rows, 15.7 min
494/564 rows, 17.9 min
514/564 rows, 19.8 min
534/564 rows, 21.7 min
554/564 rows, 23.7 min
  OOM twice on row idx 97 (len 2526) -> NaN
rows left NaN after repeated OOM: 1 -> [97]
base judge: 564 rows in 25.3 min (2.69 s/row) | NaN 1
base p_halluc deciles: [0.0, 0.01, 0.848, 1.0, 1.0]


In [6]:
# 6 — wiki-evidence arm on UNRESOLVED no-context factual rows + RUNTIME GUARD: if the
# projected finish would exceed the budget (adversarial held-out mix), cap evidence to
# top-1 chunk / 800 chars. On the public test the guard never fires (reproduction exact).
t0 = time.time()
widx = WikiIndex.load(WIKI_PARQ[0].parent)
print(f"wiki index: {len(widx.chunks)} chunks, loaded {time.time()-t0:.0f}s")

judge.PROMPT_BY_TYPE["evidence_soft"] = PROMPT_EVIDENCE
ev_mask = (test.row_type == "factual_no_context").to_numpy() & ~resolved
noctx = test[ev_mask].copy().reset_index(drop=True)

BUDGET_S = 7.0 * 3600            # trip point: cap evidence if projected > 7 h so the
                                # worst-case held-out (~7.7 h uncapped) finishes in ~5.5 h with
                                # margin. Public projects 2.1 h so the guard never fires there
                                # -> public reproduction stays bit-exact to the deployed 0.910.
EV_RATE, IDIOM_RATE = 9.5, 1.7   # measured s/row (uncapped evidence, idiom)
idiom_unres = int(((test.row_type == "idiom").to_numpy() & ~resolved).sum())
projected_s = (time.time() - T_START) + len(noctx) * EV_RATE + idiom_unres * IDIOM_RATE
EV_K, EV_CAP = 2, 1200           # the config that produced LB 0.870
if projected_s > BUDGET_S:
    EV_K, EV_CAP = 1, 800
    print(f"⚠️ RUNTIME GUARD: projected {projected_s/3600:.1f} h > {BUDGET_S/3600:.0f} h "
          f"— capping evidence to top-{EV_K} / {EV_CAP} chars")

t0 = time.time()
noctx["context"] = [evidence_context(widx.retrieve(r.prompt_bn, str(r.response_bn), k=EV_K), cap=EV_CAP)
                    for r in noctx.itertuples()]
noctx["row_type"] = "evidence_soft"
print(f"retrieval: {len(noctx)} rows in {(time.time()-t0)/60:.1f} min | "
      f"empty evidence: {int((noctx['context'].str.len() == 0).sum())}")

t0 = time.time()
ev_scores = score_rows(noctx, model, tokenizer, token_budget=2000,
                       checkpoint_path="/kaggle/working/ckpt_ev_scores.json")
print(f"evidence arm: {len(noctx)} rows in {(time.time()-t0)/60:.1f} min "
      f"| NaN: {int(ev_scores['p_halluc'].isna().sum())}")
ev_scores.to_parquet(f"/kaggle/working/evjudge_{EVCFG}_test.parquet", index=False)

wiki index: 609536 chunks, loaded 20s
retrieval: 535 rows in 1.3 min | empty evidence: 5
25/535 rows, 2.1 min
45/535 rows, 4.7 min
65/535 rows, 7.8 min
85/535 rows, 11.2 min
105/535 rows, 14.8 min
125/535 rows, 18.4 min
145/535 rows, 22.1 min
165/535 rows, 25.8 min
185/535 rows, 29.5 min
205/535 rows, 33.3 min
225/535 rows, 37.2 min
245/535 rows, 41.0 min
265/535 rows, 44.8 min
285/535 rows, 48.7 min
305/535 rows, 52.5 min
325/535 rows, 56.4 min
345/535 rows, 60.2 min
365/535 rows, 64.1 min
385/535 rows, 68.0 min
405/535 rows, 71.9 min
425/535 rows, 75.8 min
445/535 rows, 79.7 min
465/535 rows, 83.6 min
485/535 rows, 87.6 min
505/535 rows, 91.5 min
525/535 rows, 95.6 min
evidence arm: 535 rows in 97.7 min | NaN: 0


In [7]:
# 7 — idiom-corpus route on UNRESOLVED idiom rows 
lookup = load_corpus(CORPUS_JSON[0])
idi = test[(test.row_type == "idiom").to_numpy() & ~resolved].copy()
m = match_rows(idi, lookup).set_index("id")
routed = idi[idi.id.isin(m[m.matched_idiom.notna()].index)].copy()
routed["context"] = [pseudo_context(m.loc[i, "entry"]) for i in routed.id]
routed["row_type"] = "context_entailment"
print(f"idiom rows: {len(idi)} | corpus-matched (routed): {len(routed)}")

t0 = time.time()
if len(routed):
    idiom_scores = score_rows(routed, model, tokenizer, token_budget=2000,
                              checkpoint_path="/kaggle/working/ckpt_idiom_scores.json")
    idiom_scores.to_parquet(f"/kaggle/working/idiomctx_{CFG}_test.parquet", index=False)
else:
    idiom_scores = pd.DataFrame(columns=["id", "p_halluc"])
n_nan = int(idiom_scores["p_halluc"].isna().sum()) if len(idiom_scores) else 0
print(f"idiom re-scored: {len(routed)} rows in {(time.time() - t0) / 60:.1f} min | NaN: {n_nan}")

idiom rows: 74 | corpus-matched (routed): 73
67/73 rows, 2.2 min
idiom re-scored: 73 rows in 2.5 min | NaN: 0


In [8]:
# 8 — assemble (reproduces LB 0.910): base judge hard@0.5 (NaN->rules) -> evidence arm
# replaces no-ctx factual -> idiom verdicts -> safety overrides -> CPU routers OVERRIDE

sidx = test.set_index("id").index
p = test[["id"]].merge(test_scores[["id", "p_halluc"]], on="id")["p_halluc"].to_numpy()
pred = np.where(np.isnan(p), pred_rules, hard_verdicts(p))
n_nan = int(np.isnan(p).sum())

ev_ok = ev_scores[ev_scores["p_halluc"].notna()]
if len(ev_ok):
    pred[sidx.get_indexer(ev_ok["id"])] = hard_verdicts(ev_ok["p_halluc"])

if len(idiom_scores):
    keep = ~idiom_scores["p_halluc"].isna().to_numpy()
    pos = sidx.get_indexer(idiom_scores["id"])
    pred[pos[keep]] = hard_verdicts(idiom_scores["p_halluc"])[keep]

# NOTE: the sample-as-key override is intentionally NOT applied — the deployed

flips = contradiction_overrides(test, pred, p)
if flips:
    pred[sidx.get_indexer(pd.Index(flips))] = 0

# CPU routers OVERRIDE last (final Phase-1 precedence)
n_route = int((pred[resolved] != ROUTES[resolved].to_numpy().astype(int)).sum())
pred[resolved] = ROUTES[resolved].to_numpy().astype(int)

sub = pd.DataFrame({"id": test["id"].astype(int), "label": pred.astype(int)})
sub.to_csv("/kaggle/working/submission.csv", index=False)

assert list(sub.columns) == ["id", "label"], "submission must be exactly id,label"
assert sub["label"].isin([0, 1]).all() and sub["id"].is_unique
rate0 = float((sub["label"] == 0).mean())
print(f"NaN->rules {n_nan} | evidence-arm {len(ev_ok)} | idiom {int(len(idiom_scores))} | "
      f"contradiction {len(flips)} | CPU-router overrides {n_route}")
print(f"halluc-rate: {rate0:.3f}",
      "✓ inside 0.35-0.55" if 0.35 <= rate0 <= 0.55 else "⚠️ OUTSIDE 0.35-0.55 — investigate")
print("wrote /kaggle/working/submission.csv")

NaN->rules 1953 | evidence-arm 535 | idiom 73 | contradiction 0 | CPU-router overrides 363
halluc-rate: 0.496 ✓ inside 0.35-0.55
wrote /kaggle/working/submission.csv


In [9]:
# 9 — Phase-1 regression diff (skips on Phase-2 data) + total wall-clock
if REF_CSV:
    ref = pd.read_csv(REF_CSV[0])
    if len(ref) == len(sub) and (ref["id"].to_numpy() == sub["id"].to_numpy()).all():
        diff = int((ref["label"].to_numpy() != sub["label"].to_numpy()).sum())
        print(f"diff vs submission_expQ_bankkey.csv (LB 0.910): {diff} rows "
              f"({diff / len(sub):.2%}) — expect ~0-15 from CUDA nondeterminism")
    else:
        print("reference ids/length differ — held-out data, diff skipped")
else:
    print("no reference CSV attached — Phase-2 mode, diff skipped")

import subprocess
print(subprocess.run(["ls", "-la", "/kaggle/working"], capture_output=True, text=True).stdout)
print(f"TOTAL WALL-CLOCK: {(time.time() - T_START) / 3600:.2f} h (budget: 9 h)")

diff vs submission_expQ_bankkey.csv (LB 0.910): 1 rows (0.04%) — expect ~0-15 from CUDA nondeterminism
total 180
drwxr-xr-x 3 root root  4096 Jul 20 17:22 .
drwxr-xr-x 9 root root  4096 Jul 20 15:10 ..
drwxr-xr-x 2 root root  4096 Jul 20 15:10 artifacts
-rw-r--r-- 1 root root 26319 Jul 20 17:19 ckpt_ev_scores.json
-rw-r--r-- 1 root root  3493 Jul 20 17:22 ckpt_idiom_scores.json
-rw-r--r-- 1 root root 27908 Jul 20 15:40 ckpt_test_scores.json
-rw-r--r-- 1 root root 12922 Jul 20 17:19 evjudge_172f2079_test.parquet
-rw-r--r-- 1 root root  3762 Jul 20 17:22 idiomctx_2ae8d46a_test.parquet
-rw-r--r-- 1 root root 24363 Jul 20 15:40 judge_qwen3_2ae8d46a_test.parquet
---------- 1 root root 42844 Jul 20 17:22 __notebook__.ipynb
-rw-r--r-- 1 root root 16514 Jul 20 17:22 submission.csv

TOTAL WALL-CLOCK: 2.23 h (budget: 9 h)
